# Time is the Hardest Dimension

Time dimensions are special: they're naturally ordered, and they need an aggregation granularity (daily, weekly, monthly). This makes them the foundation for a family of transforms that are simple to express but non-trivial to implement in SQL.

This notebook demonstrates SLayer's time-related features with working code.

See also: [Formulas](../../concepts/formulas.md) | [Queries](../../concepts/queries.md)

**Prerequisites:** `pip install motley-slayer[examples]` (jafgen is installed by the cell below)

In [1]:
# Install jafgen (Jaffle Shop data generator) from a specific commit
# The released PyPI version has a bug; this pinned commit is the fix.
# This is only needed for running the tutorials — not a SLayer dependency.
!pip install -q git+https://github.com/rossbowen/jaffle-shop-generator.git@09557a1118b000071f8171aa97d54d5029bf0f0b


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

engine, storage, models = ensure_jaffle_shop()

## Time Dimensions and Granularity

A time dimension truncates a date/timestamp column to a given granularity. SLayer supports `day`, `week`, `month`, `quarter`, and `year`.

Let's start with monthly revenue:

In [3]:
result = engine.execute(
    query={
        "source_model": "orders",
        "time_dimensions": [{"dimension": "ordered_at", "granularity": "month"}],
        "fields": ["order_total:sum"],
        "order": [{"column": "ordered_at", "direction": "asc"}],
        "limit": 12,
    }
)

print(f"{'Month':<12} {'Revenue':>14}")
print("-" * 28)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    print(f"{month:<12} ${rev:>13,.2f}")

Month               Revenue
----------------------------
2018-09      $     5,387.60
2018-10      $     9,807.56
2018-11      $    12,766.60
2018-12      $    15,394.21
2019-01      $    20,245.35
2019-02      $    18,964.09
2019-03      $    30,481.62
2019-04      $    37,713.68
2019-05      $    40,357.84
2019-06      $    37,440.30
2019-07      $    22,340.99
2019-08      $    24,010.28


## Month-over-Month Change

The `change()` transform computes the difference from the previous period. It needs no extra arguments — the time dimension and granularity are already known from the query. Internally it uses a self-join CTE (not LAG), so it handles time gaps correctly and never produces edge NULLs.

`change_pct()` gives the percentage change:

In [4]:
result = engine.execute(
    query={
        "source_model": "orders",
        "time_dimensions": [{"dimension": "ordered_at", "granularity": "month"}],
        "fields": [
            "order_total:sum",
            {"formula": "change(order_total:sum)", "name": "mom_change"},
            {"formula": "change_pct(order_total:sum)", "name": "mom_pct"},
        ],
        "order": [{"column": "ordered_at", "direction": "asc"}],
        "limit": 12,
    }
)

print(f"{'Month':<12} {'Revenue':>14} {'MoM Change':>14} {'MoM %':>8}")
print("-" * 52)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    chg = row["orders.mom_change"]
    pct = row["orders.mom_pct"]
    chg_str = f"${chg:+,.0f}" if chg is not None else "-"
    pct_str = f"{pct:+.1%}" if pct is not None else "-"
    print(f"{month:<12} ${rev:>13,.2f} {chg_str:>14} {pct_str:>8}")

Month               Revenue     MoM Change    MoM %
----------------------------------------------------
2018-09      $     5,387.60              -        -
2018-10      $     9,807.56        $+4,420   +82.0%
2018-11      $    12,766.60        $+2,959   +30.2%
2018-12      $    15,394.21        $+2,628   +20.6%
2019-01      $    20,245.35        $+4,851   +31.5%
2019-02      $    18,964.09        $-1,281    -6.3%
2019-03      $    30,481.62       $+11,518   +60.7%
2019-04      $    37,713.68        $+7,232   +23.7%
2019-05      $    40,357.84        $+2,644    +7.0%
2019-06      $    37,440.30        $-2,918    -7.2%
2019-07      $    22,340.99       $-15,099   -40.3%
2019-08      $    24,010.28        $+1,669    +7.5%


## Year-over-Year with time_shift

`time_shift(measure, offset, granularity)` shifts the time dimension by the given offset before applying the query's granularity. This is implemented as a calendar-based self-join CTE — semantically simple, but the SQL is non-trivial.

For example, `time_shift(order_total_sum, -1, 'year')` gives "same month, previous year":

In [5]:
result = engine.execute(
    query={
        "source_model": "orders",
        "time_dimensions": [
            {
                "dimension": "ordered_at",
                "granularity": "month",
                "date_range": ["2020-01-01", "2020-12-31"],
            }
        ],
        "fields": [
            "order_total:sum",
            {"formula": "time_shift(order_total:sum, -1, 'year')", "name": "prev_year"},
        ],
        "order": [{"column": "ordered_at", "direction": "asc"}],
    }
)

print(f"{'Month':<12} {'Revenue':>14} {'Prev Year':>14}")
print("-" * 42)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    prev = row["orders.prev_year"]
    prev_str = f"${prev:>13,.2f}" if prev is not None else f"{'n/a':>14}"
    print(f"{month:<12} ${rev:>13,.2f} {prev_str}")

Month               Revenue      Prev Year
------------------------------------------
2020-01      $    77,483.46 $    20,245.35
2020-02      $    71,937.08 $    18,964.09
2020-03      $    77,138.49 $    30,481.62
2020-04      $    74,446.37 $    37,713.68
2020-05      $    98,094.45 $    40,357.84
2020-06      $    77,030.22 $    37,440.30
2020-07      $    42,683.83 $    22,340.99
2020-08      $    46,562.34 $    24,010.28
2020-09      $    76,190.18 $    36,416.40
2020-10      $   137,747.54 $    70,540.19
2020-11      $   141,252.76 $    69,639.88
2020-12      $   152,584.00 $    74,727.42


### time_shift vs lag

Without a granularity argument, `time_shift(x, -1)` is equivalent to `lag(x, 1)` — it uses SQL's `LAG()` window function to look at the previous row. This is more efficient but produces NULLs at the edges (the first row has no previous row).

With a granularity argument (like `'year'` above), `time_shift` uses a calendar-based self-join instead — no edge NULLs, and the shift can be any amount (not just a multiple of the query granularity).

## The last() Transform

`last(measure)` returns the most recent time period's value, broadcast to every row. This **collapses the time dimension** — useful for comparing each period to the latest one, or for filtering.

For example, revenue by store alongside each store's most recent month's revenue:

In [6]:
result = engine.execute(
    query={
        "source_model": "orders",
        "time_dimensions": [{"dimension": "ordered_at", "granularity": "quarter"}],
        "fields": [
            "order_total:sum",
            {"formula": "last(order_total:sum)", "name": "latest_quarter"},
        ],
        "dimensions": ["stores.name"],
        "order": [{"column": "ordered_at", "direction": "desc"}],
        "limit": 10,
    }
)

print(f"{'Quarter':<12} {'Store':<20} {'Revenue':>14} {'Latest Qtr':>14}")
print("-" * 62)
for row in result.data:
    qtr = str(row["orders.ordered_at"])[:7]
    store = row["orders.stores.name"]
    rev = row["orders.order_total_sum"]
    latest = row["orders.latest_quarter"]
    print(f"{qtr:<12} {store:<20} ${rev:>13,.2f} ${latest:>13,.2f}")

Quarter      Store                       Revenue     Latest Qtr
--------------------------------------------------------------
2021-07      Chicago              $    32,944.83 $    24,871.96
2021-07      Brooklyn             $    36,888.76 $    24,871.96
2021-07      San Francisco        $    31,271.77 $    24,871.96
2021-07      Philadelphia         $    24,871.96 $    24,871.96
2021-07      New Orleans          $    14,987.42 $    24,871.96
2021-04      Chicago              $   113,705.53 $    24,871.96
2021-04      New Orleans          $    32,308.62 $    24,871.96
2021-04      San Francisco        $   102,258.81 $    24,871.96
2021-04      Brooklyn             $   137,909.13 $    24,871.96
2021-04      Philadelphia         $    79,404.37 $    24,871.96


## Composing Transforms in Filters

Transforms can be composed and used in filters. For example, `last(change(order_total_sum)) < 0` keeps only rows where the most recent period's change was negative — i.e., revenue declined in the latest period.

This is a powerful pattern: "show me all stores, but only those whose most recent quarter-over-quarter revenue went down":

In [7]:
result = engine.execute(
    query={
        "source_model": "orders",
        "time_dimensions": [{"dimension": "ordered_at", "granularity": "quarter"}],
        "fields": [
            "order_total:sum",
            {"formula": "change(order_total:sum)", "name": "qoq_change"},
        ],
        "dimensions": ["stores.name"],
        "filters": ["last(change(order_total:sum)) < 0"],
        "order": [{"column": "ordered_at", "direction": "desc"}],
        "limit": 10,
    }
)

print("Stores where latest QoQ revenue declined:")
print(f"{'Quarter':<12} {'Store':<20} {'Revenue':>14} {'QoQ Change':>14}")
print("-" * 62)
for row in result.data:
    qtr = str(row["orders.ordered_at"])[:7]
    store = row["orders.stores.name"]
    rev = row["orders.order_total_sum"]
    chg = row["orders.qoq_change"]
    chg_str = f"${chg:+,.0f}" if chg is not None else "-"
    print(f"{qtr:<12} {store:<20} ${rev:>13,.2f} {chg_str:>14}")

Stores where latest QoQ revenue declined:
Quarter      Store                       Revenue     QoQ Change
--------------------------------------------------------------
2021-07      Brooklyn             $    36,888.76       $+21,901
2021-07      Philadelphia         $    24,871.96       $-88,834
2021-07      Chicago              $    32,944.83        $+8,073
2021-07      San Francisco        $    31,271.77        $-1,673
2021-07      New Orleans          $    14,987.42       $-21,901
2021-04      Chicago              $   113,705.53       $+81,397
2021-04      San Francisco        $   102,258.81            $+0
2021-04      Philadelphia         $    79,404.37       $-58,505
2021-04      New Orleans          $    32,308.62       $-47,096
2021-04      Brooklyn             $   137,909.13       $+49,146


## Summary

Each of these concepts is simple and natural at the semantic level, but requires non-trivial SQL (CTEs, self-joins, window functions) to implement. SLayer offloads that effort.

| Transform | What it does | SQL implementation |
|-----------|-------------|-------------------|
| `change(x)` | Difference from previous period | Self-join CTE (gap-safe, no edge NULLs) |
| `change_pct(x)` | Percentage change from previous period | Same as change, divided |
| `time_shift(x, -1, 'year')` | Same measure, shifted in time | Calendar-based self-join CTE |
| `lag(x)` / `lead(x)` | Previous/next row's value | SQL LAG/LEAD window (NULLs at edges) |
| `last(x)` | Most recent period's value, broadcast | Sub-query with FIRST_VALUE |
| `cumsum(x)` | Running total over time | SUM window function |

All transforms compose: `cumsum(change(x))`, `last(change(x))` in filters, etc.

See the [Formulas docs](../../concepts/formulas.md) for the full reference.